### COPY INTO and MERGE Commands

**COPY INTO Command**

- Incrementally load the data

In [0]:
--DROP TABLE IF EXISTS gizmobox_sivan.delta_lake.raw_stock_prices;
CREATE TABLE  gizmobox_sivan.delta_lake.raw_stock_prices;

In [0]:
--delete from gizmobox_sivan.delta_lake.raw_stock_prices;

copy into gizmobox_sivan.delta_lake.raw_stock_prices
from 'abfss://gizmoboxsivan@ourdbpracticeextdl.dfs.core.windows.net/landing/stock_prices'
FILEFORMAT = JSON
format_options('inferSchema' = 'true')
copy_options('mergeSchema' = 'true')

In [0]:
select * from gizmobox_sivan.delta_lake.raw_stock_prices;

In [0]:
create or replace table gizmobox_sivan.delta_lake.stock_prices(
    stock_id string,
    price double,
    trading_date string
)

**Merge the source data into target table**

In [0]:
merge into gizmobox_sivan.delta_lake.stock_prices as target
using gizmobox_sivan.delta_lake.raw_stock_prices as source
on target.stock_id =  source.stock_id
when matched and source.status = 'ACTIVE' then
    update set target.price = source.price, target.trading_date = source.trading_date
when matched and source.status = 'DELISTED' then    
    delete
when not matched and source.status = 'ACTIVE' then
    insert (stock_id, price, trading_date) values (source.stock_id, source.price, source.trading_date)


In [0]:
select * from gizmobox_sivan.delta_lake.stock_prices